# 23. PR-AUC가 평가 방식에 따라 얼마나 달라지나 (다른 조 0.69 재현 시도)

**고정**: 공식 2차 labeled · 초기허용불량 제외 · 원값 21개 변수
**바꿔 보는 것**
| 항목 | 경우 |
|---|---|
| 단위 | row(LH/RH 행 그대로) / shot(같은 샷 합침) |
| 분할 | 층화 무작위 5-fold / 같은 샷은 같은 fold / 시간순 5구간 / Run 하나씩 빼기(LORO) |
| PR-AUC 계산 | AP(확률, 계단식) / 사다리꼴·확률 / 사다리꼴·0/1 판정값 |
| 집계 | fold 평균 ± 표준편차 / 전체 OOF 한 번에 |
| 모델 | RF, 로지스틱 회귀(지도) + 최종 모델(비지도, 비교용) |

**쌍둥이 누수율** = 테스트 불량 행 중 같은 샷의 다른 행(공정값 완전 동일)이 학습에 들어간 비율

⏱ RF 시드 5개 반복 때문에 수 분 소요. 최종 모델 점수는 `outputs/final_model/{p}_scores.csv` 사용 (20a·20b 먼저 실행돼 있어야 함)

In [1]:
import os
import pandas as pd
from pr_split_check import run, summarize
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40); pd.set_option('display.max_rows', 100)
OUT = './outputs/pr_split_check'
os.makedirs(OUT, exist_ok=True)

F, P, LOG = run()
S = summarize(F, P)
F.to_csv(f'{OUT}/folds.csv', index=False, encoding='utf-8-sig')
S.to_csv(f'{OUT}/summary.csv', index=False, encoding='utf-8-sig')
display(LOG)

  cn7 row  random     RF
  cn7 row  random     LR
  cn7 row  random     최종모델(비지도)
  cn7 row  group_shot RF
  cn7 row  group_shot LR
  cn7 row  group_shot 최종모델(비지도)
  cn7 row  time_block RF
  cn7 row  time_block LR
  cn7 row  time_block 최종모델(비지도)
  cn7 row  loro       RF
  cn7 row  loro       LR
  cn7 row  loro       최종모델(비지도)
  cn7 shot random     RF
  cn7 shot random     LR
  cn7 shot random     최종모델(비지도)
  cn7 shot time_block RF
  cn7 shot time_block LR
  cn7 shot time_block 최종모델(비지도)
  cn7 shot loro       RF
  cn7 shot loro       LR
  cn7 shot loro       최종모델(비지도)
  rg3 row  random     RF
  rg3 row  random     LR
  rg3 row  random     최종모델(비지도)
  rg3 row  group_shot RF
  rg3 row  group_shot LR
  rg3 row  group_shot 최종모델(비지도)
  rg3 row  time_block RF
  rg3 row  time_block LR
  rg3 row  time_block 최종모델(비지도)
  rg3 row  loro       RF
  rg3 row  loro       LR
  rg3 row  loro       최종모델(비지도)
  rg3 shot random     RF
  rg3 shot random     LR
  rg3 shot random     최종모델(비지도)
  rg3 shot time_

,제품,2차 행,2차 불량 행,초기허용 제외 후 행,원값 연결 실패(제외),행,불량 행,샷,불량 샷
0,cn7,1211,17,1211,0,1211,17,606,14
1,rg3,1182,25,1182,0,1182,25,591,25


## 1. 요약 (제품 × 단위 × 분할 × 모델)

In [2]:
cols = ['제품', '단위', '분할', '모델', '평가 fold 수', '쌍둥이 누수율', 'AP (fold 평균±sd)', 'AP (전체 OOF)',
        'PR-AUC 사다리꼴·확률', 'PR-AUC 사다리꼴·0/1']
for p in ['cn7', 'rg3']:
    print(p.upper())
    display(S[S['제품'] == p][cols].round(3))

CN7


,제품,단위,분할,모델,평가 fold 수,쌍둥이 누수율,AP (fold 평균±sd),AP (전체 OOF),PR-AUC 사다리꼴·확률,PR-AUC 사다리꼴·0/1
4,cn7,row,random,LR,25,0.767,0.479 ± 0.253,0.430,0.453 ± 0.274,0.384 ± 0.154
5,cn7,row,random,RF,25,0.767,0.432 ± 0.194,0.407,0.411 ± 0.198,0.395 ± 0.227
6,cn7,row,random,최종모델(비지도),25,0.767,0.371 ± 0.234,0.373,0.365 ± 0.236,0.508 ± 0.235
0,cn7,row,group_shot,LR,25,0.000,0.441 ± 0.283,0.403,0.422 ± 0.310,0.424 ± 0.174
1,cn7,row,group_shot,RF,25,0.000,0.425 ± 0.276,0.380,0.404 ± 0.288,0.523 ± 0.193
2,cn7,row,group_shot,최종모델(비지도),25,0.000,0.356 ± 0.284,0.373,0.350 ± 0.287,0.555 ± 0.251
7,cn7,row,time_block,최종모델(비지도),1,0.000,0.426 ± nan,0.373,0.423 ± nan,0.699 ± nan
3,cn7,row,loro,최종모델(비지도),1,0.000,0.413 ± nan,0.373,0.411 ± nan,0.691 ± nan
9,cn7,shot,random,LR,25,0.000,0.453 ± 0.235,0.349,0.410 ± 0.249,0.391 ± 0.151
10,cn7,shot,random,RF,25,0.000,0.419 ± 0.235,0.281,0.363 ± 0.244,0.452 ± 0.166


RG3


,제품,단위,분할,모델,평가 fold 수,쌍둥이 누수율,AP (fold 평균±sd),AP (전체 OOF),PR-AUC 사다리꼴·확률,PR-AUC 사다리꼴·0/1
19,rg3,row,random,LR,25,0.832,0.027 ± 0.011,0.019,0.020 ± 0.007,0.164 ± 0.116
20,rg3,row,random,RF,25,0.832,0.035 ± 0.019,0.025,0.026 ± 0.012,0.011 ± 0.000
21,rg3,row,random,최종모델(비지도),25,0.832,0.035 ± 0.015,0.024,0.026 ± 0.009,0.111 ± 0.204
13,rg3,row,group_shot,LR,25,0.000,0.025 ± 0.006,0.019,0.019 ± 0.004,0.152 ± 0.099
14,rg3,row,group_shot,RF,25,0.000,0.043 ± 0.030,0.026,0.037 ± 0.042,0.511 ± 0.000
15,rg3,row,group_shot,최종모델(비지도),25,0.000,0.034 ± 0.012,0.024,0.025 ± 0.008,0.291 ± 0.253
22,rg3,row,time_block,LR,5,0.000,0.023 ± 0.013,0.015,0.018 ± 0.010,0.126 ± 0.134
23,rg3,row,time_block,RF,5,0.000,0.048 ± 0.051,0.022,0.056 ± 0.080,0.511 ± 0.005
24,rg3,row,time_block,최종모델(비지도),5,0.000,0.039 ± 0.027,0.024,0.030 ± 0.019,0.411 ± 0.225
16,rg3,row,loro,LR,4,0.000,0.021 ± 0.003,0.016,0.017 ± 0.002,0.224 ± 0.184


## 2. 0.69에 가장 가까운 조합
fold 평균 AP·사다리꼴(확률·0/1) 중 0.69에 가까운 순

In [3]:
import numpy as np
long = F.melt(id_vars=['제품', '단위', '분할', '모델'], value_vars=['AP', 'trapz_prob', 'trapz_bin'],
              var_name='계산법', value_name='값')
near = (long.groupby(['제품', '단위', '분할', '모델', '계산법'])['값'].agg(['mean', 'std']).reset_index()
            .assign(차이=lambda d: (d['mean'] - 0.69).abs()).sort_values('차이').head(10))
display(near.round(3))

,제품,단위,분할,모델,계산법,mean,std,차이
10,cn7,row,loro,최종모델(비지도),trapz_bin,0.691,NaN,0.001
22,cn7,row,time_block,최종모델(비지도),trapz_bin,0.699,NaN,0.009
37,cn7,shot,time_block,최종모델(비지도),trapz_bin,0.652,NaN,0.038
25,cn7,shot,loro,최종모델(비지도),trapz_bin,0.636,NaN,0.054
7,cn7,row,group_shot,최종모델(비지도),trapz_bin,0.555,0.251,0.135
79,rg3,shot,loro,RF,trapz_bin,0.523,0.004,0.167
4,cn7,row,group_shot,RF,trapz_bin,0.523,0.193,0.167
97,rg3,shot,time_block,RF,trapz_bin,0.521,0.010,0.169
88,rg3,shot,random,RF,trapz_bin,0.521,0.000,0.169
52,rg3,row,loro,RF,trapz_bin,0.512,0.002,0.178


## 해석 가이드
- **row · random**의 쌍둥이 누수율이 높고 점수가 크게 오르면 → 행 단위 무작위 분할의 누수가 원인
- **row · random → row · group_shot**에서 떨어지는 폭 = 쌍둥이 누수 효과
- **group_shot → time_block / loro**에서 떨어지는 폭 = 같은 시간대(불량 몰림)를 외우는 효과
- **사다리꼴·0/1**이 AP보다 크면 → 판정값으로 PR-AUC를 계산해 생기는 과대
- 최종 모델은 학습이 없어 분할과 무관하게 점수가 같아야 함 (fold 평균만 fold 구성에 따라 달라짐)